# Imports

Installs

In [74]:
# ! pip install -U ipython-sql prettytable==3.9.0

Load extension

In [75]:
%load_ext sql

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


Load data

In [76]:
%sql sqlite:///data.sqlite

Get table names 

In [77]:
%%sql
SELECT name 
FROM sqlite_master 
WHERE type='table';

 * sqlite:///data.sqlite
Done.


name
orderdetails
payments
offices
customers
orders
productlines
products
employees


# Grouping Data with SQL

## Introduction to GROUP BY and Aggregate Functions

Aggregate functions in SQL are used to summarize data. They help calculate values like totals, averages, minimums, and maximums.

Common aggregate functions:

- `COUNT()` – Counts rows  
- `MIN()` – Returns the smallest value  
- `MAX()` – Returns the largest value  
- `SUM()` – Returns the total  
- `AVG()` – Returns the average  

The `GROUP BY` clause groups rows into summary rows and returns one result per group. It is typically used together with aggregate functions.

Example use case:
- Counting customers per country
- Comparing total sales across regions

In [78]:
%%sql
SELECT COUNT(*) AS employeeCount FROM employees

 * sqlite:///data.sqlite
Done.


employeeCount
23


In [79]:
%%sql
select * from products limit 2

 * sqlite:///data.sqlite
Done.


productCode,productName,productLine,productScale,productVendor,productDescription,quantityInStock,buyPrice,MSRP
S10_1678,1969 Harley Davidson Ultimate Chopper,Motorcycles,1:10,Min Lin Diecast,"This replica features working kickstand, front suspension, gear-shift lever, footbrake lever, drive chain, wheels and steering. All parts are particularly delicate due to their precise scale and require special care and attention.",7933,48.81,95.70
S10_1949,1952 Alpine Renault 1300,Classic Cars,1:10,Classic Metal Creations,Turnable front wheels; steering function; detailed interior; detailed engine; opening hood; opening trunk; opening doors; and detailed chassis.,7305,98.58,214.30


In [80]:
%%sql
SELECT SUM(quantityInStock) AS stockCount FROM products 

 * sqlite:///data.sqlite
Done.


stockCount
555131


Most expensive  product

In [81]:
%%sql
SELECT *,MAX(MSRP)  FROM products

 * sqlite:///data.sqlite
Done.


productCode,productName,productLine,productScale,productVendor,productDescription,quantityInStock,buyPrice,MSRP,MAX(MSRP)
S32_1374,1997 BMW F650 ST,Motorcycles,1:32,Exoto Designs,Features official die-struck logos and baked enamel finish. Comes with stand.,178,66.92,99.89,99.89


Most Profitable product 

In [82]:
%%sql
SELECT
  *,  max(MSRP - buyPrice) AS max_profit
FROM
  products

 * sqlite:///data.sqlite
Done.


productCode,productName,productLine,productScale,productVendor,productDescription,quantityInStock,buyPrice,MSRP,max_profit
S10_1949,1952 Alpine Renault 1300,Classic Cars,1:10,Classic Metal Creations,Turnable front wheels; steering function; detailed interior; detailed engine; opening hood; opening trunk; opening doors; and detailed chassis.,7305,98.58,214.30,115.72000000000001


## What is GROUP BY?

`GROUP BY` is used in SQL to **combine rows that have the same values in one or more columns** into summary rows.

It is usually used together with **aggregate functions**

## Basic Syntax

```sql
SELECT column_name, AGGREGATE_FUNCTION(column_name)
FROM table_name
GROUP BY column_name;

In [83]:
%%sql
SELECT country,Count(country)
FROM customers
GROUP BY country
LIMIT 5;

 * sqlite:///data.sqlite
Done.


country,Count(country)
Australia,5
Austria,2
Belgium,2
Canada,3
Denmark,2


In [84]:
%%sql
SELECT * FROM orders LIMIT 5

 * sqlite:///data.sqlite
Done.


orderNumber,orderDate,requiredDate,shippedDate,status,comments,customerNumber
10100,2003-01-06,2003-01-13,2003-01-10,Shipped,,363
10101,2003-01-09,2003-01-18,2003-01-11,Shipped,Check on availability.,128
10102,2003-01-10,2003-01-18,2003-01-14,Shipped,,181
10103,2003-01-29,2003-02-07,2003-02-02,Shipped,,121
10104,2003-01-31,2003-02-09,2003-02-01,Shipped,,141


In [85]:
%%sql
SELECT customerNumber,COUNT(orderNumber) FROM orders GROUP BY customerNumber Limit 5

 * sqlite:///data.sqlite
Done.


customerNumber,COUNT(orderNumber)
103,3
112,3
114,5
119,4
121,4


In [86]:
%%sql
SELECT * FROM customers LIMIT 5

 * sqlite:///data.sqlite
Done.


customerNumber,customerName,contactLastName,contactFirstName,phone,addressLine1,addressLine2,city,state,postalCode,country,salesRepEmployeeNumber,creditLimit
103,Atelier graphique,Schmitt,Carine,40.32.2555,"54, rue Royale",,Nantes,,44000,France,1370,21000.00
112,Signal Gift Stores,King,Jean,7025551838,8489 Strong St.,,Las Vegas,NV,83030,USA,1166,71800.00
114,"Australian Collectors, Co.",Ferguson,Peter,03 9520 4555,636 St Kilda Road,Level 3,Melbourne,Victoria,3004,Australia,1611,117300.00
119,La Rochelle Gifts,Labrune,Janine,40.67.8555,"67, rue des Cinquante Otages",,Nantes,,44000,France,1370,118200.00
121,Baane Mini Imports,Bergulfsen,Jonas,07-98 9555,Erling Skakkes gate 78,,Stavern,,4110,Norway,1504,81700.00


Count Number of customers in each city

In [87]:
%%sql
SELECT city, COUNT (customerNumber) FROM customers
GROUP BY city LIMIT 5

 * sqlite:///data.sqlite
Done.


city,COUNT (customerNumber)
Aachen,1
Allentown,1
Amsterdam,1
Auckland,1
Auckland,2



## Alternative GROUP BY Syntax

Instead of grouping by a column name, you can group by the column position in the `SELECT` statement.

Example:

```sql
SELECT country, COUNT(*)
FROM customers
GROUP BY 1;
```

`GROUP BY 1` refers to the first selected column (`country`).



In [88]:
%%sql
SELECT country,city,COUNT(city)
FROM customers
GROUP BY 2 LIMIT 10;

 * sqlite:///data.sqlite
Done.


country,city,COUNT(city)
Germany,Aachen,1
USA,Allentown,1
Netherlands,Amsterdam,1
New Zealand,Auckland,1
New Zealand,Auckland,2
Spain,Barcelona,1
Italy,Bergamo,1
Norway,Bergen,1
Germany,Berlin,1
Switzerland,Bern,1


## Aliasing

An alias is a temporary name given to a column or table.

Example:

```sql
SELECT country, COUNT(*) AS customer_count
FROM customers
GROUP BY country;
```

Important notes:

- Aliases only exist during the query.
- The keyword `AS` is optional in SQLite.
- Aliases improve readability, especially when using aggregates.


In [89]:
%%sql
SELECT country, COUNT(*) AS customer_count
FROM customers
GROUP BY country LIMIT 5;


 * sqlite:///data.sqlite
Done.


country,customer_count
Australia,5
Austria,2
Belgium,2
Canada,3
Denmark,2


## Multiple Aggregations Example

We calculated payment statistics per customer:

```sql
SELECT
    customerNumber,
    COUNT(*) AS number_payments,
    MIN(amount) AS min_purchase,
    MAX(amount) AS max_purchase,
    AVG(amount) AS avg_purchase,
    SUM(amount) AS total_spent
FROM payments
GROUP BY customerNumber;
```

This returned:

- Number of payments  
- Minimum purchase  
- Maximum purchase  
- Average purchase  
- Total spent  

All grouped by `customerNumber`.


In [90]:
%%sql
SELECT
    customerNumber,
    COUNT(*) AS number_payments,
    MIN(CAST(amount AS INTEGER)) AS min_purchase,
    MAX(CAST(amount AS INTEGER)) AS max_purchase,
    AVG(CAST(amount AS INTEGER)) AS avg_purchase,
    SUM(CAST(amount AS INTEGER)) AS total_spent
FROM payments
GROUP BY customerNumber LIMIT 5;

 * sqlite:///data.sqlite
Done.


customerNumber,number_payments,min_purchase,max_purchase,avg_purchase,total_spent
103,3,1676,14571,7437.666666666667,22313
112,3,14191,33347,26726.333333333332,80179
114,4,7565,82261,45146.0,180584
119,3,19501,49523,38982.666666666664,116948
121,4,1491,50218,26055.75,104223



## Filtering with WHERE (Before Aggregation)

The `WHERE` clause filters rows before grouping happens.

Example: Filtering payments made in 2004.

```sql
SELECT
    customerNumber,
    COUNT(*) AS number_payments,
    MIN(amount) AS min_purchase,
    MAX(amount) AS max_purchase,
    AVG(amount) AS avg_purchase,
    SUM(amount) AS total_spent
FROM payments
WHERE strftime('%Y', paymentDate) = '2004'
GROUP BY customerNumber;
```

Key points:

- Filtering happens before aggregation.
- Results change because only filtered rows are included.
- Columns used in `WHERE` do not need to appear in `SELECT`.


In [91]:
%%sql
SELECT
    customerNumber,
    COUNT(*) AS number_payments,
    MIN(CAST(amount AS INTEGER)) AS min_purchase,
    MAX(CAST(amount AS INTEGER)) AS max_purchase,
    AVG(CAST(amount AS INTEGER)) AS avg_purchase,
    SUM(CAST(amount AS INTEGER)) AS total_spent
FROM payments
WHERE strftime('%Y', paymentDate) = '2004'
GROUP BY customerNumber LIMIT 5;

 * sqlite:///data.sqlite
Done.


customerNumber,number_payments,min_purchase,max_purchase,avg_purchase,total_spent
103,2,1676,6066,3871.0,7742
112,2,14191,33347,23769.0,47538
114,2,44894,82261,63577.5,127155
119,2,19501,47924,33712.5,67425
121,2,17876,34638,26257.0,52514


In [92]:
%%sql
SELECT * FROM offices

 * sqlite:///data.sqlite
Done.


officeCode,city,phone,addressLine1,addressLine2,state,country,postalCode,territory
1,San Francisco,+1 650 219 4782,100 Market Street,Suite 300,CA,USA,94080,NA
2,Boston,+1 215 837 0825,1550 Court Place,Suite 102,MA,USA,02107,NA
3,NYC,+1 212 555 3000,523 East 53rd Street,apt. 5A,NY,USA,10022,NA
4,Paris,+33 14 723 4404,43 Rue Jouffroy D'abbans,,,France,75017,EMEA
5,Tokyo,+81 33 224 5000,4-1 Kioicho,,Chiyoda-Ku,Japan,102-8578,Japan
6,Sydney,+61 2 9264 2451,5-11 Wentworth Avenue,Floor #2,,Australia,NSW 2010,APAC
7,London,+44 20 7877 2041,25 Old Broad Street,Level 7,,UK,EC2N 1HN,EMEA


In [93]:
%%sql
select country,count(officeCode) officeCount from offices where country != "UK" group by country

 * sqlite:///data.sqlite
Done.


country,officeCount
Australia,1
France,1
Japan,1
USA,3


## The HAVING Clause (After Aggregation)

The `HAVING` clause filters results after grouping and aggregation.

Example:

```sql
SELECT
    customerNumber,
    COUNT(*) AS number_payments,
    MIN(CAST(amount AS INTEGER)) AS min_purchase,
    MAX(CAST(amount AS INTEGER)) AS max_purchase,
    AVG(CAST(amount AS INTEGER)) AS avg_purchase,
    SUM(CAST(amount AS INTEGER)) AS total_spent
FROM payments
GROUP BY customerNumber
HAVING AVG(amount) > 50000;
```

Key difference:

- `WHERE` → Filters rows before grouping  
- `HAVING` → Filters aggregated results after grouping  

In most SQL dialects, you must repeat the aggregate function in `HAVING` instead of using its alias.


In [94]:
%%sql
SELECT
    customerNumber,
    COUNT(*) AS number_payments,
    MIN(CAST(amount AS INTEGER)) AS min_purchase,
    MAX(CAST(amount AS INTEGER)) AS max_purchase,
    AVG(CAST(amount AS INTEGER)) AS avg_purchase,
    SUM(CAST(amount AS INTEGER)) AS total_spent
FROM payments
GROUP BY customerNumber
HAVING COUNT(*) <= 2 LIMIT 5;

 * sqlite:///data.sqlite
Done.


customerNumber,number_payments,min_purchase,max_purchase,avg_purchase,total_spent
144,2,7674,36005,21839.5,43679
157,2,35152,63357,49254.5,98509
167,2,12538,85024,48781.0,97562
171,2,18997,42783,30890.0,61780
173,2,11843,20355,16099.0,32198


Return only countries with more than 1 office 

In [95]:
%%sql
SELECT country,count(officeCode) officeCount 
FROM offices 
WHERE country != "USA" 
GROUP BY country 
HAVING count(officeCode) > 1

 * sqlite:///data.sqlite
Done.


country,officeCount


## Combining WHERE and HAVING

We can combine both clauses for more complex logic.

Example: Customers who made at least 2 purchases over 50,000.

```sql
SELECT
    customerNumber,
    COUNT(*) AS number_payments,
    MIN(CAST(amount AS INTEGER)) AS min_purchase,
    MAX(CAST(amount AS INTEGER)) AS max_purchase,
    AVG(CAST(amount AS INTEGER)) AS avg_purchase,
    SUM(CAST(amount AS INTEGER)) AS total_spent
FROM payments
WHERE amount > 50000
GROUP BY customerNumber
HAVING COUNT(*) >= 2;
```

Process:

1. `WHERE` filters payments over 50,000  
2. `GROUP BY` aggregates by customer  
3. `HAVING` filters customers with at least 2 such payments  


In [96]:
%%sql
SELECT
    customerNumber,
    COUNT(*) AS number_payments,
    MIN(CAST(amount AS INTEGER)) AS min_purchase,
    MAX(CAST(amount AS INTEGER)) AS max_purchase,
    AVG(CAST(amount AS INTEGER)) AS avg_purchase,
    SUM(CAST(amount AS INTEGER)) AS total_spent
FROM payments
WHERE amount > 50000
GROUP BY customerNumber
HAVING COUNT(*) >= 2 LIMIT 5;

 * sqlite:///data.sqlite
Done.


customerNumber,number_payments,min_purchase,max_purchase,avg_purchase,total_spent
103,3,1676,14571,7437.666666666667,22313
112,3,14191,33347,26726.333333333332,80179
114,4,7565,82261,45146.0,180584
119,3,19501,49523,38982.666666666664,116948
121,4,1491,50218,26055.75,104223


## Using ORDER BY and LIMIT

We can also sort and limit aggregated results.

Example:

```sql
SELECT
    customerNumber,
    COUNT(*) AS number_payments,
    MIN(amount) AS min_purchase,
    MAX(amount) AS max_purchase,
    AVG(amount) AS avg_purchase,
    SUM(amount) AS total_spent
FROM payments
WHERE amount > 50000
GROUP BY customerNumber
HAVING COUNT(*) >= 2
ORDER BY total_spent
LIMIT 1;
```

This finds the customer with the lowest total spending among those meeting the criteria.


In [97]:
%%sql
SELECT
    customerNumber,
    COUNT(*) AS number_payments,
    MIN(amount) AS min_purchase,
    MAX(amount) AS max_purchase,
    AVG(amount) AS avg_purchase,
    SUM(amount) AS total_spent
FROM payments
WHERE amount > 50000
GROUP BY customerNumber
HAVING COUNT(*) >= 2
ORDER BY total_spent
LIMIT 1;

 * sqlite:///data.sqlite
Done.


customerNumber,number_payments,min_purchase,max_purchase,avg_purchase,total_spent
219,2,3452.75,4465.85,3959.3,7918.6


# ORDER BY

By default, SQL returns rows in the order they are stored in the database. This order may not be meaningful.

To sort results, we use:

```sql
SELECT column(s)
FROM table_name
ORDER BY column_name;
```

Example:

```sql
SELECT *
FROM products
ORDER BY productName;
```

In [98]:
%%sql
SELECT *
FROM products
LIMIT 2;

 * sqlite:///data.sqlite
Done.


productCode,productName,productLine,productScale,productVendor,productDescription,quantityInStock,buyPrice,MSRP
S10_1678,1969 Harley Davidson Ultimate Chopper,Motorcycles,1:10,Min Lin Diecast,"This replica features working kickstand, front suspension, gear-shift lever, footbrake lever, drive chain, wheels and steering. All parts are particularly delicate due to their precise scale and require special care and attention.",7933,48.81,95.70
S10_1949,1952 Alpine Renault 1300,Classic Cars,1:10,Classic Metal Creations,Turnable front wheels; steering function; detailed interior; detailed engine; opening hood; opening trunk; opening doors; and detailed chassis.,7305,98.58,214.30


In [99]:
%%sql
SELECT *
FROM products
ORDER BY productName LIMIT 2;

 * sqlite:///data.sqlite
Done.


productCode,productName,productLine,productScale,productVendor,productDescription,quantityInStock,buyPrice,MSRP
S18_3136,18th Century Vintage Horse Carriage,Vintage Cars,1:18,Red Start Diecast,"Hand crafted diecast-like metal horse carriage is re-created in about 1:18 scale of antique horse carriage. This antique style metal Stagecoach is all hand-assembled with many different parts.This collectible metal horse carriage is painted in classic Red, and features turning steering wheel and is entirely hand-finished.",5992,60.74,104.72
S24_2011,18th century schooner,Ships,1:24,Carousel DieCast Legends,"All wood with canvas sails. Many extras including rigging, long boats, pilot house, anchors, etc. Comes with 4 masts, all square-rigged.",1898,82.34,122.89


## Ascending and Descending Order

- `ASC` → Ascending order (default)
- `DESC` → Descending order

Example:

```sql
ORDER BY productName ASC;
ORDER BY productName DESC;
```


In [100]:
%%sql
SELECT *
FROM products
ORDER BY productName ASC LIMIT 2;

 * sqlite:///data.sqlite
Done.


productCode,productName,productLine,productScale,productVendor,productDescription,quantityInStock,buyPrice,MSRP
S18_3136,18th Century Vintage Horse Carriage,Vintage Cars,1:18,Red Start Diecast,"Hand crafted diecast-like metal horse carriage is re-created in about 1:18 scale of antique horse carriage. This antique style metal Stagecoach is all hand-assembled with many different parts.This collectible metal horse carriage is painted in classic Red, and features turning steering wheel and is entirely hand-finished.",5992,60.74,104.72
S24_2011,18th century schooner,Ships,1:24,Carousel DieCast Legends,"All wood with canvas sails. Many extras including rigging, long boats, pilot house, anchors, etc. Comes with 4 masts, all square-rigged.",1898,82.34,122.89


In [101]:
%%sql
SELECT *
FROM products
ORDER BY productName DESC LIMIT 2;

 * sqlite:///data.sqlite
Done.


productCode,productName,productLine,productScale,productVendor,productDescription,quantityInStock,buyPrice,MSRP
S700_2610,The USS Constitution Ship,Ships,1:700,Red Start Diecast,"All wood with canvas sails. Measures 31 1/2"" Length x 22 3/8"" High x 8 1/4"" Width. Extras include 4 boats on deck, sea sprite on bow, anchors, copper railing, pilot houses, etc.",7083,33.97,72.28
S700_3505,The Titanic,Ships,1:700,Carousel DieCast Legends,"Completed model measures 19 1/2 inches long, 9 inches high, 3inches wide and is in barn red/black. All wood and metal.",1956,51.09,100.17



# Sorting by Multiple Columns

You can sort by more than one column.

Example:

```sql
SELECT productVendor, productName, MSRP
FROM products
ORDER BY productVendor, productName;
```

How it works:
1. Sort by `productVendor`
2. If vendors are the same, use `productName` as a tiebreaker

Important rule:
- Columns with fewer unique values should come first.
- Columns with more unique values should come later.

In [102]:
%%sql
SELECT productVendor, productName, MSRP
FROM products
ORDER BY productVendor, productName LIMIT 5;

 * sqlite:///data.sqlite
Done.


productVendor,productName,MSRP
Autoart Studio Design,1900s Vintage Bi-Plane,68.51
Autoart Studio Design,1932 Model A Ford J-Coupe,127.13
Autoart Studio Design,1937 Horch 930V Limousine,65.75
Autoart Studio Design,1962 Volkswagen Microbus,127.79
Autoart Studio Design,1968 Ford Mustang,194.57


We can check uniqueness using:

```sql
SELECT COUNT(DISTINCT productVendor),
       COUNT(DISTINCT productName)
FROM products;
```

In [103]:
%%sql
SELECT COUNT(DISTINCT productVendor),
       COUNT(DISTINCT productName)
FROM products

 * sqlite:///data.sqlite
Done.


COUNT(DISTINCT productVendor),COUNT(DISTINCT productName)
13,110



# Type Casting in ORDER BY

Sometimes data appears numeric but is stored as text.

Example problem:

```sql
ORDER BY quantityInStock;
```

If `quantityInStock` is stored as text, it will sort alphabetically instead of numerically.

Solution: Use `CAST`

```sql
ORDER BY CAST(quantityInStock AS INTEGER);
```

This forces proper numeric sorting.


In [106]:
%%sql
SELECT *
FROM products
ORDER BY quantityInStock LIMIT 20

 * sqlite:///data.sqlite
Done.


productCode,productName,productLine,productScale,productVendor,productDescription,quantityInStock,buyPrice,MSRP
S24_1046,1970 Chevy Chevelle SS 454,Classic Cars,1:24,Unimax Art Galleries,"This model features rotating wheels, working streering system and opening doors. All parts are particularly delicate due to their precise scale and require special care and attention. It should not be picked up by the doors, roof, hood or trunk.",1005,49.24,73.49
S50_1392,Diamond T620 Semi-Skirted Tanker,Trucks and Buses,1:50,Highway 66 Mini Classics,This limited edition model is licensed and perfectly scaled for Lionel Trains. The Diamond T620 has been produced in solid precision diecast and painted with a fire baked enamel finish. It comes with a removable tanker and is a perfect model to add authenticity to your static train or car layout or to just have on display.,1016,68.29,115.75
S12_3891,1969 Ford Falcon,Classic Cars,1:12,Second Gear Diecast,Turnable front wheels; steering function; detailed interior; detailed engine; opening hood; opening trunk; opening doors; and detailed chassis.,1049,83.05,173.02
S18_4721,1957 Corvette Convertible,Classic Cars,1:18,Classic Metal Creations,1957 die cast Corvette Convertible in Roman Red with white sides and whitewall tires. 1:18 scale quality die-cast with detailed engine and underbvody. Now you can own The Classic Corvette.,1249,69.93,148.80
S32_4289,1928 Ford Phaeton Deluxe,Vintage Cars,1:32,Highway 66 Mini Classics,"This model features grille-mounted chrome horn, lift-up louvered hood, fold-down rumble seat, working steering system",136,33.02,68.79
S24_2887,1952 Citroen-15CV,Classic Cars,1:24,Exoto Designs,"Precision crafted hand-assembled 1:18 scale reproduction of the 1952 15CV, with its independent spring suspension, working steering system, opening doors and hood, detailed engine and instrument panel, all topped of with a factory fresh baked enamel finish.",1452,72.82,117.44
S24_2000,1960 BSA Gold Star DBD34,Motorcycles,1:24,Highway 66 Mini Classics,Detailed scale replica with working suspension and constructed from over 70 parts,15,37.32,76.17
S12_1666,1958 Setra Bus,Trucks and Buses,1:12,Welly Diecast Productions,"Model features 30 windows, skylights & glare resistant glass, working steering system, original logos",1579,77.90,136.67
S50_1514,1962 City of Detroit Streetcar,Trains,1:50,Classic Metal Creations,"This streetcar is a joy to see. It has 99 separate windows, electric wire guides, detailed interiors with seats, poles and drivers controls, rolling and turning wheel assemblies, plus authentic factory baked-enamel finishes (Green Hornet for Chicago and Cream and Crimson for Boston).",1645,37.49,58.58
S32_1374,1997 BMW F650 ST,Motorcycles,1:32,Exoto Designs,Features official die-struck logos and baked enamel finish. Comes with stand.,178,66.92,99.89


In [107]:
%%sql
SELECT *
FROM products
ORDER BY CAST(quantityInStock AS INTEGER) LIMIT 20

 * sqlite:///data.sqlite
Done.


productCode,productName,productLine,productScale,productVendor,productDescription,quantityInStock,buyPrice,MSRP
S24_2000,1960 BSA Gold Star DBD34,Motorcycles,1:24,Highway 66 Mini Classics,Detailed scale replica with working suspension and constructed from over 70 parts,15,37.32,76.17
S12_1099,1968 Ford Mustang,Classic Cars,1:12,Autoart Studio Design,"Hood, doors and trunk all open to reveal highly detailed interior features. Steering wheel actually turns the front wheels. Color dark green.",68,95.34,194.57
S32_4289,1928 Ford Phaeton Deluxe,Vintage Cars,1:32,Highway 66 Mini Classics,"This model features grille-mounted chrome horn, lift-up louvered hood, fold-down rumble seat, working steering system",136,33.02,68.79
S32_1374,1997 BMW F650 ST,Motorcycles,1:32,Exoto Designs,Features official die-struck logos and baked enamel finish. Comes with stand.,178,66.92,99.89
S72_3212,Pont Yacht,Ships,1:72,Unimax Art Galleries,"Measures 38 inches Long x 33 3/4 inches High. Includes a stand.Many extras including rigging, long boats, pilot house, anchors, etc. Comes with 2 masts, all square-rigged",414,33.30,54.60
S18_2248,1911 Ford Town Car,Vintage Cars,1:18,Motor City Art Classics,"Features opening hood, opening doors, opening trunk, wide white wall tires, front door arm rests, working steering system.",540,33.30,60.54
S18_2795,1928 Mercedes-Benz SSK,Vintage Cars,1:18,Gearbox Collectibles,"This 1:18 replica features grille-mounted chrome horn, lift-up louvered hood, fold-down rumble seat, working steering system, chrome-covered spare, opening doors, detailed and wired engine. Color black.",548,72.56,168.75
S700_3167,F/A 18 Hornet 1/72,Planes,1:72,Motor City Art Classics,"10"" Wingspan with retractable landing gears.Comes with pilot",551,54.40,80.00
S50_4713,2002 Yamaha YZR M1,Motorcycles,1:50,Autoart Studio Design,"Features rotating wheels , working kick stand. Comes with stand.",600,34.17,81.36
S700_1938,The Mayflower,Ships,1:700,Studio M Art Models,"Measures 31 1/2 inches Long x 25 1/2 inches High x 10 5/8 inches WideAll wood with canvas sail. Extras include long boats, rigging, ladders, railing, anchors, side cannons, hand painted, etc.",737,43.30,86.61


Product with most stock

In [108]:
%%sql
SELECT *
FROM products
ORDER BY CAST(quantityInStock AS INTEGER) DESC LIMIT 1

 * sqlite:///data.sqlite
Done.


productCode,productName,productLine,productScale,productVendor,productDescription,quantityInStock,buyPrice,MSRP
S12_2823,2002 Suzuki XREO,Motorcycles,1:12,Unimax Art Galleries,"Official logos and insignias, saddle bags located on side of motorcycle, detailed engine, working steering, working suspension, two leather seats, luggage rack, dual exhaust pipes, small saddle bag located on handle bars, two-tone paint with chrome accents, superior die-cast detail , rotating wheels , working kick stand, diecast metal with plastic parts and baked enamel finish.",9997,66.27,150.62
